# MANAS: Multilingual Emotion Classifier Fine-Tuning (MuRIL / XLM-RoBERTa)

This Google Colab notebook fine-tunes a multilingual transformer (`google/muril-base-cased` or `xlm-roberta-base`) on GoEmotions augmented with synthetic Indian-language affective samples (Hindi, Tamil, Telugu, Kannada, Malayalam, English).

### Pipeline Steps:
1. **Environment Setup & Dependency Installation**
2. **Dataset Acquisition & Multi-Label Mapping** (GoEmotions + Indic samples)
3. **Tokenization & Stratified Train/Val Split (80/20)**
4. **Training with AdamW & Linear Warmup**
5. **Evaluation: Per-Class Precision, Recall, and F1 Metrics**
6. **ONNX Export & Dynamic Quantization for Fast Production Pre-Check**

In [ ]:
# Step 1: Install dependencies
!pip install -q transformers datasets evaluate accelerate onnx onnxruntime scikit-learn

In [ ]:
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using compute device: {device}')

In [ ]:
# Step 2: Define Target Emotion Taxonomy
EMOTION_LABELS = ['sadness', 'anxiety', 'anger', 'overwhelm', 'grief', 'hope', 'neutral']
id2label = {i: label for i, label in enumerate(EMOTION_LABELS)}
label2id = {label: i for i, label in enumerate(EMOTION_LABELS)}

# Map GoEmotions fine-grained labels to MANAS wellness categories
GOEMOTIONS_MAPPING = {
    'sadness': 'sadness', 'disappointment': 'sadness', 'remorse': 'sadness',
    'fear': 'anxiety', 'nervousness': 'anxiety',
    'anger': 'anger', 'annoyance': 'anger', 'disgust': 'anger',
    'grief': 'grief',
    'optimism': 'hope', 'relief': 'hope', 'joy': 'hope', 'gratitude': 'hope',
    'neutral': 'neutral'
}

print('Loading GoEmotions from HuggingFace Hub...')
raw_goemotions = load_dataset('go_emotions', 'simplified', split='train')
print(f'Loaded {len(raw_goemotions)} GoEmotions records.')

In [ ]:
# Step 3: Augment with Multilingual Indic Wellness Utterances
indic_samples = [
    {'text': 'எனக்கு ரொம்ப பயமா இருக்கு எக்ஸாம் நினைச்சா கை நடுங்குது', 'label': 'anxiety'},
    {'text': 'மனசே உடைஞ்சு போச்சு என்னால தாங்க முடியல', 'label': 'sadness'},
    {'text': 'வேலை பளு அதிகமா இருக்கு உடம்பும் மனசும் முடியல', 'label': 'overwhelm'},
    {'text': 'मुझे बहुत घबराहट हो रही है दिल की धड़कन तेज है', 'label': 'anxiety'},
    {'text': 'सब खत्म हो गया कुछ अच्छा नहीं लग रहा', 'label': 'sadness'},
    {'text': 'काम का बहुत तनाव है बिल्कुल हिम्मत नहीं बची', 'label': 'overwhelm'},
    {'text': 'నాకు చాలా భయంగా ఉంది పరీక్షల వల్ల తట్టుకోలేకపోతున్నాను', 'label': 'anxiety'},
    {'text': 'జీవితం మీద ఆశ పోయింది చాలా ఒంటరిగా ఉంది', 'label': 'sadness'},
    {'text': 'ಪರೀಕ್ಷೆ ಭಯದಿಂದ ನನಗೆ ನಿದ್ದೆ ಬರ್ತಿಲ್ಲ ತಲೆ ಸುತ್ತುತ್ತಿದೆ', 'label': 'anxiety'},
    {'text': 'എനിക്ക് വല്ലാത്ത ഭയവും വിറയലും തോന്നുന്നു', 'label': 'anxiety'}
]

indic_df = pd.DataFrame(indic_samples)
print(f'Indic synthetic set loaded: {len(indic_df)} samples.')

In [ ]:
# Step 4: Tokenizer & Preprocessing
MODEL_CKPT = 'google/muril-base-cased'  # Alternative: 'xlm-roberta-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=len(EMOTION_LABELS),
    id2label=id2label,
    label2id=label2id
)
model.to(device)
print(f'Initialized {MODEL_CKPT} for {len(EMOTION_LABELS)} emotion classes.')

In [ ]:
# Step 5: Evaluation Metric Function (Per-class F1, Macro F1, Weighted F1)
f1_metric = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    macro_f1 = f1_metric.compute(predictions=predictions, references=labels, average='macro')['f1']
    weighted_f1 = f1_metric.compute(predictions=predictions, references=labels, average='weighted')['f1']
    per_class_f1 = f1_metric.compute(predictions=predictions, references=labels, average=None)['f1']
    
    res = {'macro_f1': macro_f1, 'weighted_f1': weighted_f1}
    for idx, score in enumerate(per_class_f1):
        res[f'f1_{id2label[idx]}'] = round(float(score), 4)
    return res

In [ ]:
# Step 6: Export to ONNX for Production Pre-Check
dummy_input = tokenizer('I feel very overwhelmed and stressed', return_tensors='pt').to(device)
onnx_export_path = 'emotion_model.onnx'

torch.onnx.export(
    model,
    (dummy_input['input_ids'], dummy_input['attention_mask']),
    onnx_export_path,
    input_names=['input_ids', 'attention_mask'],
    output_names=['logits'],
    dynamic_axes={
        'input_ids': {0: 'batch_size', 1: 'sequence_length'},
        'attention_mask': {0: 'batch_size', 1: 'sequence_length'},
        'logits': {0: 'batch_size'}
    },
    opset_version=14
)
print(f'Successfully exported model to ONNX: {onnx_export_path}')

# Save tokenizer
tokenizer.save_pretrained('tokenizer_export')
print('Tokenizer saved to tokenizer_export/')